# 使用ONNX导出一个简单的 MNIST MLP

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import os
os.environ['TORCHVISION_MNIST_URL'] = 'https://mirrors.aliyun.com/torchvision/datasets/mnist/'
# 超简单的 MLP 网络
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)  # 展平
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

def main():
    # 数据预处理
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    # 使用本地 MNIST
    train_dataset = datasets.MNIST(
        root=r"C:/Users/27427/Desktop/code/AI_infer_learn/MNIST",
        train=True, download=True, transform=transform
    )
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

    # 初始化模型
    device = torch.device("cuda")
    model = MLP().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # 简单训练几个 epoch
    model.train()
    for epoch in range(2):  # 跑 2 个 epoch 够导出测试
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, loss={loss.item():.4f}")

    # 导出到 ONNX
    dummy_input = torch.randn(1, 1, 28, 28, device=device)
    torch.onnx.export(
        model, dummy_input, "model.onnx",
        input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
        opset_version=12
    )
    print("✅ 模型已导出到 model.onnx")

if __name__ == "__main__":
    main()


100.0%
100.0%
100.0%
100.0%


Epoch 1, loss=0.0617
Epoch 2, loss=0.1319
✅ 模型已导出到 model.onnx


C:\Users\27427\AppData\Local\Temp\ipykernel_18568\3917782390.py:58: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


### 核心方法：`torch.onnx.export`

PyTorch 官方提供了 `torch.onnx.export` 函数来完成这个转换过程。其核心思想是**执行一次模型的前向传播（推理）**，并**跟踪记录所有操作**，从而构建一个计算图，最终将其保存为 ONNX 格式。

#### 基本步骤：

1.  **训练并准备好你的 PyTorch 模型**：确保模型处于评估模式（`eval()`），而不是训练模式。在训练模式下Dropout 层的神经元会被随机关闭，这是为了防止过拟合。
2.  **创建一个示例输入（Dummy Input）**：这个输入张量的形状和类型应该与模型在实际推理时接受的输入一致。
3.  **调用 `torch.onnx.export` 函数**：提供模型、示例输入以及希望保存的 ONNX 文件名。
4.  **(可选) 检查导出的 ONNX 模型**：使用 `onnx` 库或在线工具（如 Netron）来验证模型结构是否正确。

---

### 详细代码示例

假设我们有一个简单的预训练图像分类模型（如 ResNet-50）。

```python
import torch
import torchvision

# 1. 加载/创建并准备模型
model = torchvision.models.resnet50(pretrained=True) # 示例模型

# 将模型设置为评估模式至关重要！
# 这会禁用 dropout、BatchNorm 的更新等训练特定操作。
model.eval()

# 2. 创建示例输入（ dummy input ）
# 维度通常是 (batch_size, channels, height, width)
# 类型应与您未来推理时使用的输入一致（通常是 float32）
batch_size = 1
dummy_input = torch.randn(batch_size, 3, 224, 224) 

# 3. 指定导出模型的动态轴（可选，但对于可变输入尺寸很重要）
# 我们希望 batch_size 和 图像尺寸 可以是动态的，而不是固定的 1, 224, 224
# 定义动态轴字典：
#   - 第一个参数是输入的名称（对于单输入模型，通常是 0 或者你定义的名字）
#   - 字典的 key 是维度索引，value 是维度的名称
dynamic_axes = {
    'input': {0: 'batch_size', 2: 'height', 3: 'width'},  # 为输入张量定义动态维度
    'output': {0: 'batch_size'},  # 为输出张量定义动态维度
}

# 4. 导出模型
input_names = ['input']   # 给输入节点起个名字
output_names = ['output'] # 给输出节点起个名字

# 核心导出命令
torch.onnx.export(
    model,                  # 要导出的模型
    dummy_input,            # 模型输入（可以是一个元组或多个参数）
    "resnet50_dynamic.onnx", # 导出的 ONNX 文件名
    export_params=True,     # 将模型参数也一并导出
    opset_version=14,       # 使用的 ONNX 算子集版本（推荐>=11）
    do_constant_folding=True, # 是否执行常量折叠优化
    input_names=input_names,   # 指定输入的名称
    output_names=output_names, # 指定输出的名称
    dynamic_axes=dynamic_axes  # 指定动态维度
)

print("Model has been converted to ONNX.")
```

#### 参数解释：

*   `model`: 要转换的 PyTorch 模型。
*   `args`: 模型的输入。可以是张量，也可以是元组。
*   `f`: 导出的 ONNX 文件的路径（如 `"my_model.onnx"`）。
*   `export_params` (bool): 如果为 `True`，则导出所有模型参数（权重）。如果要导出的是一个无参数的模型结构，则设为 `False`。
*   `opset_version` (int): ONNX 算子的版本。版本越高，支持的算子越多。建议使用 11 或更高版本，以获得更完整的算子支持（如对于 `Resize` 操作）。
*   `do_constant_folding` (bool): 是否进行常量折叠优化。`True` 通常会产生更小、更高效的模型。
*   `input_names` (list of str): 为计算图的输入节点命名，便于后续识别。
*   `output_names` (list of str): 为计算图的输出节点命名。
*   `dynamic_axes` (dict): **非常重要**，用于指定允许变化的输入/输出维度（动态批处理大小、动态图像尺寸等）。如果不指定，所有维度都将被固定为示例输入的大小。

---

### 验证导出的 ONNX 模型

导出后，强烈建议进行验证，确保转换正确无误。

**方法一：使用 ONNX Runtime 进行推理验证**

这是最可靠的验证方法，可以检查模型是否能被其他推理引擎正确加载和运行。

```python
import onnxruntime
import numpy as np

# 加载 ONNX 模型
onnx_model_path = "resnet50_dynamic.onnx"
ort_session = onnxruntime.InferenceSession(onnx_model_path)

# 准备输入数据（格式为 numpy array）
dummy_input_np = dummy_input.numpy()

# ONNX Runtime 运行推理
# 注意：input_name 必须与导出时设置的 ‘input_names' 一致
ort_inputs = {ort_session.get_inputs()[0].name: dummy_input_np}
ort_outs = ort_session.run(None, ort_inputs)

# 用原始 PyTorch 模型运行推理，得到参考输出
with torch.no_grad():
    torch_outs = model(dummy_input)

# 比较 ONNX Runtime 和 PyTorch 的输出结果
# 因为精度转换，可能会有微小误差，我们检查是否在可接受范围内
np.testing.assert_allclose(torch_outs.numpy(), ort_outs[0], rtol=1e-03, atol=1e-05)
print("Exported model has been tested with ONNXRuntime, and the result looks good!")
```

---

### 常见问题与技巧 (Troubleshooting)

1.  **动态轴（Dynamic Axes）**：如果你的模型需要处理**可变大小的输入**（如不同的批处理大小、不同分辨率的图像），**必须**通过 `dynamic_axes` 参数明确指定哪些维度是动态的。这是导出过程中最常见的问题来源。
2.  **算子不支持 (Unsupported operator)**：
    *   **错误信息**：可能提示 `RuntimeError: ONNX export failed: Couldn't export operator <operator_name>`。
    *   **解决方案**：
        *   尝试升级 `torch` 和 `onnx` 到最新版本。
        *   尝试使用更高版本的 `opset_version` (如 13, 14, 15)。
        *   对于一些不常见的或自定义的算子，你可能需要自己为 ONNX 实现并注册该算子，这是一个更高级的话题。
3.  **跟踪（Tracing）与脚本（Scripting）**：
    *   `torch.onnx.export` 默认使用**跟踪（Tracing）** 模式，它记录一条具体输入路径上的操作。如果模型中有控制流（if/else, for loop），跟踪模式可能会失败，因为它只记录了一条分支。
    *   对于包含控制流的模型，需要使用 **TorchScript** 的**脚本（Scripting）** 模式。你可以先用 `torch.jit.script` 编译你的模型，然后再导出。
    ```python
    scripted_model = torch.jit.script(model)
    torch.onnx.export(scripted_model, ...)
    ```
4.  **输入/输出名称**：给输入输出起一个有意义的名称（如 `'input_image'`, `'class_probabilities'`）而不是默认的，会在后续部署时（如在 TensorRT 或 OpenVINO 中）带来很大便利。